# rag-hallu-bench: open-weight detectors on a Colab GPU

Scores every open-weight detector of the benchmark on the same 5,246 rows the API detectors are scored on, and writes one `.jsonl` file per detector to Google Drive.

**How to use it**
1. *Runtime > Change runtime type* > **A100** (the 4-8B judges need a 40 GB GPU; on a smaller GPU they are skipped and only the small verifiers run).
2. *Runtime > Run all*. Allow access to Google Drive when asked.
3. The first table shows what is already on Drive. Finished detectors are skipped. Every missing one first runs on 20 rows outside the results folder, and the notebook stops with a message if those look wrong (errors, constant scores, unreadable verdicts).
4. The last cell lists the files to copy from Drive (`MyDrive/rag-hallu-bench/raw`) into `results/raw/` on your machine; then run `hallubench evaluate` there.

A session that dies can be restarted with *Run all*: runs resume from the last saved row. Everything from scratch takes several hours on an A100, most of it Lynx; a single missing detector takes minutes to about an hour.

In [ ]:
# Settings
REPO = 'https://github.com/mauroforlin/rag-hallu-bench.git'
BRANCH = 'meddecider'
EXPECTED_SHA = 'df950765b366f89c'  # printed by `hallubench build-data` on every machine
DRIVE_DIR = '/content/drive/MyDrive/rag-hallu-bench'

SMALL = ['hhem', 'minicheck-ft5', 'minicheck-deberta', 'alignscore-large', 'lettucedetect-large', 'laya']
LARGE = ['bespoke-minicheck-7b', 'granite-guardian-8b', 'biodecision-v2-4b', 'lynx-8b']
# MedDecider runs on plain transformers + PEFT, after the vLLM models. Under 80 GB the 27B and 31B load in 8-bit.
# The 31B is on Gemma 4, which is gated: accept its licence on the Hub and add an HF_TOKEN secret (key icon, left)
MEDDECIDER = ['meddecider-4b', 'meddecider-9b', 'meddecider-27b', 'meddecider-31b']

# True: score MedDecider only, skipping every other detector and the latency cell, whatever is on Drive
ONLY_MEDDECIDER = True
if ONLY_MEDDECIDER:
    SMALL, LARGE = [], []

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
# Colab secrets are readable only from the kernel, not from the `hallubench run` subprocesses:
# copy HF_TOKEN into the environment they inherit (needed for the gated Gemma base of meddecider-31b)
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('No HF_TOKEN secret (or no notebook access): gated models such as meddecider-31b will fail to download')
RESULTS = f'{DRIVE_DIR}/raw'
os.makedirs(RESULTS, exist_ok=True)
%cd /content
!rm -rf /content/rag-hallu-bench
!git clone -q -b {BRANCH} {REPO} /content/rag-hallu-bench
%cd /content/rag-hallu-bench
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# vLLM brings its own torch: install it first (the version the study ran with), then the package
!pip install -q vllm==0.31.0
!pip install -q -e ".[gpu]"
# Colab preinstalls torchao 0.10, and peft refuses to load any LoRA adapter (MedDecider) next to a
# torchao older than 0.16. Nothing here uses torchao.
!pip uninstall -q -y torchao
# Colab's preinstalled torchvision is built for Colab's torch, not vLLM's, and
# transformers fails to import its models when the two disagree. torchvision
# 0.(N+15).P goes with torch 2.N.P.
import subprocess
def py(code):
    return subprocess.run(['python', '-c', code], capture_output=True, text=True)
if py('import torchvision').returncode != 0:
    major, minor, patch = py('import torch; print(torch.__version__.split("+")[0])').stdout.strip().split('.')[:3]
    print(f'torchvision does not match torch {major}.{minor}.{patch}: installing torchvision 0.{int(minor) + 15}.{patch}')
    !pip install -q --no-deps torchvision==0.{int(minor) + 15}.{patch}
check = py('import torch, torchvision; from transformers import RobertaModel; print(torch.__version__, torchvision.__version__)')
print(check.stdout, check.stderr[-2000:])
assert check.returncode == 0, 'torch / torchvision / transformers still do not import together: send this output.'

In [ ]:
import subprocess, sys
sys.path.insert(0, '/content/rag-hallu-bench')
!rm -rf results/raw && mkdir -p results && ln -s {RESULTS} results/raw
out = subprocess.run(['hallubench', 'build-data'], capture_output=True, text=True)
print(out.stdout[-1500:], out.stderr[-1500:] if out.returncode else '')
assert EXPECTED_SHA in out.stdout, 'The rows built here differ from the reference ones: stop and check the dataset revisions.'
print('Data OK: same rows as everywhere else.')

## Helpers and current state

In [ ]:
import json, shutil
from pathlib import Path
import numpy as np
import torch
from hallubench import data
from hallubench.detectors import REGISTRY
from hallubench.runner import load_results

EXAMPLES = data.load_examples()
LABEL = {e.id: e.label for e in EXAMPLES}
SMOKE = Path('/content/smoke')
BIG_GPU = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 38e9

def eligible(name):
    spec = REGISTRY[name]
    return [e for e in EXAMPLES if (not spec.datasets or e.dataset in spec.datasets) and (not spec.core_only or e.core)]

def missing(name):
    done = {i for i, r in load_results(name, Path(RESULTS)).items() if r.get('score') is not None}
    return sum(e.id not in done for e in eligible(name))

def status():
    for name in SMALL + LARGE + MEDDECIDER:
        m, n = missing(name), len(eligible(name))
        print(f"{name:24s} {n - m:5d}/{n} rows  {'done' if m == 0 else 'MISSING ' + str(m)}")

def clear_model_cache():
    # Colab's disk holds about three of the large models at once
    shutil.rmtree(os.path.expanduser('~/.cache/huggingface/hub'), ignore_errors=True)

def smoke(name, rows=20):
    """Score ~20 rows outside the results folder and refuse to go on if they look broken."""
    shutil.rmtree(SMOKE, ignore_errors=True)
    os.environ['HALLUBENCH_RAW'] = str(SMOKE)
    try:
        !hallubench run {name} --limit {rows}
    finally:
        os.environ.pop('HALLUBENCH_RAW', None)
    res = list(load_results(name, SMOKE).values())
    for r in res:
        print(LABEL[r['id']], r['score'], (r['error'] or '')[:80], r['id'][:45])
    errors = [r for r in res if r['score'] is None]
    scores = np.array([r['score'] for r in res if r['score'] is not None], dtype=float)
    problems = []
    if not res:
        problems.append('no rows were scored')
    if errors:
        problems.append(f'{len(errors)} rows failed: {errors[0]["error"]}')
    if len(scores) > 1 and scores.std() < 1e-3:
        problems.append('every row got the same score')
    if len(scores) and np.mean(np.abs(scores - 0.5) < 1e-3) > 0.2:
        problems.append('many scores are exactly 0.5, the fallback for an unreadable verdict')
    if problems:
        raise RuntimeError(f'{name}: check failed - ' + '; '.join(problems))
    ys = np.array([LABEL[r['id']] for r in res if r['score'] is not None])
    print(f'{name}: check passed. Mean score on hallucinated rows {scores[ys == 1].mean():.2f}, '
          f'on faithful rows {scores[ys == 0].mean():.2f} (20 rows: a hint, not a result)')

def run_detector(name):
    if missing(name) == 0:
        print(f'{name}: already complete, skipped')
        return
    smoke(name)
    !hallubench run {name}
    left = missing(name)
    clear_model_cache()
    if left:
        raise RuntimeError(f'{name}: {left} rows still missing or failed; run this cell again to retry them')
    print(f'{name}: complete')

status()

## Small verifiers (any GPU)

In [ ]:
for name in SMALL:
    run_detector(name)

## 4-8B judges (A100 or larger)

One process per model, so vLLM frees the GPU before the next one loads. Lynx writes its reasoning before the verdict and is by far the slowest.

In [ ]:
if not BIG_GPU:
    print('This GPU has less than 40 GB: the 4-8B judges are skipped. Switch the runtime to A100 to run them.')
else:
    for name in LARGE:
        run_detector(name)

## MedDecider (A100 or larger)

Loaded with its authors' `meddecider_infer.py` from the pinned adapter repo, one row at a time, two forward passes per row (both option orders). On a 40 GB A100 the 27B and 31B are loaded in 8-bit, each row records its precision; on an 80 GB GPU everything runs in bf16 as its authors run it. The 27B and 31B take several hours each: if the session dies, *Run all* resumes.

In [ ]:
if not BIG_GPU:
    print('This GPU has less than 40 GB: MedDecider is skipped.')
else:
    for name in MEDDECIDER:
        run_detector(name)

## Optional: single-request latency

The full runs batch many answers per GPU call, so their recorded time is throughput, not what one user waits. This scores 50 answers one at a time for the batched detectors, into `latency/` on Drive, skipping those that already have it.

In [ ]:
if ONLY_MEDDECIDER:
    print('ONLY_MEDDECIDER: latency cell skipped')
else:
    LAT = f'{DRIVE_DIR}/latency'
    os.makedirs(LAT, exist_ok=True)
    for name in ['hhem', 'minicheck-ft5', 'minicheck-deberta', 'alignscore-large'] + (
            ['granite-guardian-8b', 'bespoke-minicheck-7b', 'lynx-8b'] if BIG_GPU else []):
        if os.path.exists(f'{LAT}/{name}.jsonl'):
            continue
        os.environ['HALLUBENCH_RAW'] = LAT
        !hallubench run {name} --limit 50 --workers 1
        os.environ.pop('HALLUBENCH_RAW', None)
        clear_model_cache()

## Done

In [ ]:
status()
print()
print(f'Copy these files from Drive ({RESULTS}) into results/raw/ on your machine, then run `hallubench evaluate`:')
for name in SMALL + LARGE + MEDDECIDER:
    p = Path(RESULTS) / f'{name}.jsonl'
    if p.exists():
        print(' ', p.name)